# BLM5135 — Track 1 training runbook

Run cell-by-cell or use **Runtime → Run all**. Outputs go directly to your Drive at `/content/drive/MyDrive/dl_project_outputs/`, so if the session dies you can come back and resume any run.

**Before starting:** confirm the runtime is A100 via *Runtime → Change runtime type → A100 GPU*. If only T4/V100/L4 is available, training will work but be slower than the budget assumes.

## 1. Mount Drive and confirm dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
DATASET_ROOT = '/content/drive/MyDrive/dataset'
JSON_PATH = f'{DATASET_ROOT}/dataset.json'
assert os.path.isfile(JSON_PATH), f'dataset.json not found at {JSON_PATH}'
with open(JSON_PATH) as fh:
    nimg = len(json.load(fh)['images'])
print(f'Drive mounted. dataset.json has {nimg} samples.')

## 2. Confirm GPU is A100 (or at least a CUDA device)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Clone the repo (or update if already cloned) and set up output dirs

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/amirkiarafiei/deep-learning-project.git'
REPO_DIR = '/content/deep-learning-project'
DRIVE_OUT = '/content/drive/MyDrive/dl_project_outputs'
HF_CACHE = f'{DRIVE_OUT}/hf_cache'

if os.path.isdir(f'{REPO_DIR}/.git'):
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, REPO_DIR], check=True)

for d in [
    DRIVE_OUT, HF_CACHE,
    f'{DRIVE_OUT}/results/track1/object',
    f'{DRIVE_OUT}/results/track1/event',
    f'{DRIVE_OUT}/results/track1/attribute',
    f'{DRIVE_OUT}/results/track1/multitask',
    f'{DRIVE_OUT}/results/track1/object_smoke',
]:
    os.makedirs(d, exist_ok=True)

subprocess.run(['git', 'log', '-1', '--oneline'], cwd=REPO_DIR)
print(f'Repo at {REPO_DIR}, outputs at {DRIVE_OUT}, HF cache at {HF_CACHE}.')

## 4. Install Python deps

Colab usually has a CUDA torch preinstalled — we only need to add `timm` and a few small libs.

In [ ]:
!pip install -q 'timm==1.0.27' 'scikit-learn>=1.4,<2.0' 'PyYAML>=6.0,<7.0' 'tqdm>=4.66'
import torch, timm, sklearn
print('torch  ', torch.__version__, '| cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')
print('timm   ', timm.__version__)
print('sklearn', sklearn.__version__)

## 5. Smoke test — 200 train / 100 val, 2 epochs, batch 32

Should finish in well under 5 minutes on A100. Confirms the full pipeline (data loader, model forward, autocast bf16, loss, optimizer, checkpoint write to Drive) works end-to-end before committing to the 4 real runs.

In [ ]:
import os, subprocess
env = os.environ.copy()
env['HF_HOME'] = HF_CACHE
env['PYTHONUNBUFFERED'] = '1'  # stream subprocess stdout to the cell live
cmd = [
    'python', '-u', 'train.py',
    '--config', 'configs/track1_object.yaml',
    '--output-dir', f'{DRIVE_OUT}/results/track1/object_smoke',
    '--dataset-root', DATASET_ROOT,
    '--json-path', JSON_PATH,
    '--run-name', 'object_smoke',
    '--subset-train', '200',
    '--subset-val', '100',
    '--epochs', '2',
    '--batch-size', '32',
    '--num-workers', '2',
]
subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)
print('\n✅ Smoke test complete.')

## 6. Full Track-1 training — four configs in sequence

Each: ConvNeXt-Tiny + 30 epochs + early stop on val avg macro-F1 + patience 10. Outputs to Drive every epoch so any session loss bounds the cost to one epoch (resume with `--resume <output_dir>/checkpoints/last.pt`).

**Run order:** object → event → attribute → multitask. Each is independent.

In [ ]:
import os, subprocess, time
env = os.environ.copy()
env['HF_HOME'] = HF_CACHE
env['PYTHONUNBUFFERED'] = '1'  # stream subprocess stdout to the cell live

RUNS = ['object', 'event', 'attribute', 'multitask']
for fam in RUNS:
    print(f'\n=== Training: {fam} ===', flush=True)
    t0 = time.time()
    out_dir = f'{DRIVE_OUT}/results/track1/{fam}'
    cmd = [
        'python', '-u', 'train.py',
        '--config', f'configs/track1_{fam}.yaml',
        '--output-dir', out_dir,
        '--dataset-root', DATASET_ROOT,
        '--json-path', JSON_PATH,
        '--num-workers', '2',
    ]
    last_pt = f'{out_dir}/checkpoints/last.pt'
    if os.path.isfile(last_pt):
        print(f'   (resuming from {last_pt})', flush=True)
        cmd += ['--resume', last_pt]
    subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)
    print(f'=== {fam} done in {(time.time() - t0)/60:.1f} min ===', flush=True)

print('\n✅ All four trainings complete.')

## 7. Evaluation on the test split + qualitative figures

Writes per-class CSVs, summary JSON, full prediction matrix, and ≥20 qualitative example PNGs + a combined PDF per run.

In [ ]:
import os, subprocess
env = os.environ.copy()
env['HF_HOME'] = HF_CACHE
env['PYTHONUNBUFFERED'] = '1'

for fam in ['object', 'event', 'attribute', 'multitask']:
    out_dir = f'{DRIVE_OUT}/results/track1/{fam}'
    ckpt = f'{out_dir}/checkpoints/best.pt'
    if not os.path.isfile(ckpt):
        print(f'(skip {fam}: no best.pt)', flush=True)
        continue
    print(f'\n=== eval+viz: {fam} ===', flush=True)
    for script, extra in [('eval.py', ['--split', 'test']),
                          ('visualize.py', ['--split', 'test', '--num-samples', '20'])]:
        subprocess.run(
            ['python', '-u', script,
             '--config', f'configs/track1_{fam}.yaml',
             '--ckpt', ckpt,
             '--output-dir', out_dir,
             '--dataset-root', DATASET_ROOT,
             '--json-path', JSON_PATH] + extra,
            cwd=REPO_DIR, env=env, check=True,
        )

print('\n✅ Eval + visualize complete. Check the per-run folders under', f'{DRIVE_OUT}/results/track1/')

## 8. Pin requirements for submission

Captures the actual library versions used in this Colab session into `requirements.txt`. This is what the course rubric requires.

Copy the output into the local repo's `requirements.txt` and commit before zipping the submission.

In [ ]:
!pip freeze | grep -iE '^(torch|torchvision|timm|scikit-learn|PyYAML|tqdm|numpy|matplotlib|Pillow|requests)=='